In [25]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Input, Dense, Reshape, Conv1DTranspose, BatchNormalization, Conv1D, LeakyReLU, Flatten, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.losses import mse
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import backend as K

In [26]:
import numpy as np

# Load
X_all = np.load('EEG_all_epochs.npy')

print("Original shape:", X_all.shape)

# Normalize per sample (VERY important for VAE)
X_all = (X_all - X_all.mean(axis=1, keepdims=True)) / \
        (X_all.std(axis=1, keepdims=True) + 1e-8)




print("Final shape:", X_all.shape)

Original shape: (4514, 512)
Final shape: (4514, 512)


In [27]:
# 80:20 training-validation split
X_train, X_test = train_test_split(X_all,  test_size=0.2)

In [28]:
# VAE model
input_shape = (512, 1)
batch_size = 32
latent_dim = 16   # was 2 → too small
epochs = 1000

# reparameterization
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=K.shape(z_mean))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon


# ======================
# Encoder
# ======================

inputs = Input(shape=input_shape, name='encoder_input')
x = inputs

# Block 1
x = Conv1D(32, kernel_size=25, strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.2)(x)

# Block 2
x = Conv1D(64, kernel_size=25, strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.2)(x)

# Block 3
x = Conv1D(128, kernel_size=15, strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = LeakyReLU(alpha=0.2)(x)

shape_before_flatten = K.int_shape(x)

x = Flatten()(x)
x = Dense(64, activation='relu')(x)

z_mean = Dense(latent_dim, name='z_mean')(x)
z_log_var = Dense(latent_dim, name='z_log_var')(x)

z = Lambda(sampling, name='z')([z_mean, z_log_var])

encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
encoder.summary()

Model: "encoder"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 encoder_input (InputLayer)     [(None, 512, 1)]     0           []                               
                                                                                                  
 conv1d_12 (Conv1D)             (None, 256, 32)      832         ['encoder_input[0][0]']          
                                                                                                  
 batch_normalization_12 (BatchN  (None, 256, 32)     128         ['conv1d_12[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 leaky_re_lu_6 (LeakyReLU)      (None, 256, 32)      0           ['batch_normalization_12[0]

In [29]:
from keras.layers import UpSampling1D

# ======================
# Decoder
# ======================

latent_inputs = Input(shape=(latent_dim,), name='z_sampling')

# Expand back to conv shape
x = Dense(shape_before_flatten[1] * shape_before_flatten[2],
          activation='relu')(latent_inputs)

x = Reshape((shape_before_flatten[1],
             shape_before_flatten[2]))(x)

# Block 1
x = UpSampling1D(size=2)(x)
x = Conv1D(128, 15, padding='same', activation='relu')(x)
x = BatchNormalization()(x)

# Block 2
x = UpSampling1D(size=2)(x)
x = Conv1D(64, 25, padding='same', activation='relu')(x)
x = BatchNormalization()(x)

# Block 3
x = UpSampling1D(size=2)(x)
x = Conv1D(32, 25, padding='same', activation='relu')(x)
x = BatchNormalization()(x)

# Final output layer
outputs = Conv1D(1, 3, padding='same', name='decoder_output')(x)

decoder = Model(latent_inputs, outputs, name='decoder')
decoder.summary()

Model: "decoder"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_sampling (InputLayer)     [(None, 16)]              0         
                                                                 
 dense_5 (Dense)             (None, 8192)              139264    
                                                                 
 reshape_2 (Reshape)         (None, 64, 128)           0         
                                                                 
 up_sampling1d_6 (UpSampling  (None, 128, 128)         0         
 1D)                                                             
                                                                 
 conv1d_15 (Conv1D)          (None, 128, 128)          245888    
                                                                 
 batch_normalization_15 (Bat  (None, 128, 128)         512       
 chNormalization)                                          

In [30]:
# VAE model (merging encoder and decoder)
outputs = decoder(encoder(inputs)[2])
vae = Model(inputs, outputs, name='vae')
vae.summary()

Model: "vae"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 encoder_input (InputLayer)  [(None, 512, 1)]          0         
                                                                 
 encoder (Functional)        [(None, 16),              702432    
                              (None, 16),                        
                              (None, 16)]                        
                                                                 
 decoder (Functional)        (None, 512, 1)            642241    
                                                                 
Total params: 1,344,673
Trainable params: 1,343,777
Non-trainable params: 896
_________________________________________________________________


In [31]:
reconstruction_loss = mse(K.flatten(inputs), K.flatten(outputs))
reconstruction_loss *= np.prod(input_shape)

# KL Divergence loss
kl_loss = 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
kl_loss = K.sum(kl_loss, axis=-1)
kl_loss *= -0.5

# Total loss
vae_loss = K.mean(reconstruction_loss + kl_loss)
vae.add_loss(vae_loss)

# Optimizer and Compile
optimizer = Adam(learning_rate=0.001, beta_1=0.5, beta_2=0.999)
vae.compile(optimizer=optimizer)

vae.summary()

Model: "vae"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 encoder_input (InputLayer)     [(None, 512, 1)]     0           []                               
                                                                                                  
 encoder (Functional)           [(None, 16),         702432      ['encoder_input[0][0]']          
                                 (None, 16),                                                      
                                 (None, 16)]                                                      
                                                                                                  
 decoder (Functional)           (None, 512, 1)       642241      ['encoder[0][2]']                
                                                                                                

In [32]:
# early stopping callback
callbacks = EarlyStopping(monitor = 'val_loss',
                          mode='min',
                          patience =50,
                          verbose = 1,
                          restore_best_weights = True)

In [ ]:
# fit vae model
history = vae.fit(X_train,X_train,
            epochs=1000,
            batch_size=32,
            validation_data=(X_test, X_test),callbacks=callbacks)

Epoch 1/1000
113/113 [==============================] - 20s 109ms/step - loss: 589.1251 - val_loss: 602.9841
Epoch 2/1000
113/113 [==============================] - 12s 107ms/step - loss: 367.0854 - val_loss: 499.8541
Epoch 3/1000
113/113 [==============================] - 13s 113ms/step - loss: 312.2828 - val_loss: 421.3979
Epoch 4/1000
113/113 [==============================] - 13s 120ms/step - loss: 276.2269 - val_loss: 342.7998
Epoch 5/1000
113/113 [==============================] - 14s 126ms/step - loss: 250.2642 - val_loss: 283.5867
Epoch 6/1000
113/113 [==============================] - 15s 130ms/step - loss: 231.0895 - val_loss: 256.2010
Epoch 7/1000
113/113 [==============================] - 15s 134ms/step - loss: 216.0870 - val_loss: 245.9534
Epoch 8/1000
113/113 [==============================] - 16s 138ms/step - loss: 203.2747 - val_loss: 236.9736
Epoch 9/1000
113/113 [==============================] - 16s 144ms/step - loss: 191.2217 - val_loss: 230.1493
Epoch 10/1000
113/1

In [ ]:
# loss curves
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('loss curves')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.show()

In [ ]:
# 2D plot of the classes in latent space
z_m, _, _ = encoder.predict(X_test,batch_size=batch_size)
plt.figure(figsize=(12, 10))
plt.scatter(z_m[:, 0], z_m[:, 1], c=X_test[:,0,0,0])
plt.xlabel("z[0]")
plt.ylabel("z[1]")
plt.show()

In [ ]:
# predicting on validation data
pred=vae.predict(X_test)

In [ ]:
# observing generated signals
plt.plot(X_test[0,:,:,0])
plt.plot(pred[0,:,:,0])